# ☕ Fine-tuning en español de `datificate/gpt2-small-spanish` (tema: Café)
**Objetivo:** Entrenar por *fine-tuning* un GPT-2 en español sobre un dominio específico (café/barismo) para observar diferencias **antes vs. después** del ajuste, y **guardar** el nuevo modelo.

**Resumen del flujo:**
1. Construcción de corpus (Wikipedia ES filtrada por café/barismo).  
2. Preprocesamiento y tokenización.  
3. Entrenamiento causal LM (sin MLM).  
4. Métricas (perplejidad) y comparaciones de generación.  
5. Guardado local y (opcional) publicación en Hugging Face Hub.


## 🔧 Instalación de dependencias


In [1]:
# !pip install -U transformers datasets accelerate sentencepiece huggingface_hub evaluate

## ⚙️ Configuración inicial y utilidades


In [2]:
import os, math, random, textwrap, numpy as np, torch

In [3]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = 0 if torch.cuda.is_available() else -1
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())


PyTorch: 2.10.0+cu128 | CUDA: True


## 🧠 Modelo base y tokenizador


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [5]:
BASE_MODEL = "datificate/gpt2-small-spanish"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL)
base_model.resize_token_embeddings(len(tokenizer))
print("Pad token:", tokenizer.pad_token, "| EOS token:", tokenizer.eos_token)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/620 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/510M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/510M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: datificate/gpt2-small-spanish
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Pad token: <|endoftext|> | EOS token: <|endoftext|>


## 📚 Construcción del corpus (tema: café / barismo)


In [6]:
from datasets import load_dataset
WIKI_CONFIG = "20231101.es"
wiki = load_dataset("wikimedia/wikipedia", WIKI_CONFIG, split="train")

wiki

README.md: 0.00B [00:00, ?B/s]

20231101.es/train-00000-of-00013.parquet:   0%|          | 0.00/688M [00:00<?, ?B/s]

20231101.es/train-00001-of-00013.parquet:   0%|          | 0.00/376M [00:00<?, ?B/s]

20231101.es/train-00002-of-00013.parquet:   0%|          | 0.00/287M [00:00<?, ?B/s]

20231101.es/train-00003-of-00013.parquet:   0%|          | 0.00/245M [00:00<?, ?B/s]

...

20231101.es/train-00010-of-00013.parquet:   0%|          | 0.00/167M [00:00<?, ?B/s]

20231101.es/train-00011-of-00013.parquet:   0%|          | 0.00/254M [00:00<?, ?B/s]

20231101.es/train-00012-of-00013.parquet:   0%|          | 0.00/226M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1841155 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'url', 'title', 'text'],
    num_rows: 1841155
})

### 🔎 Filtrado por palabras clave del dominio


In [7]:
KEYWORDS = [
    "café", "caf\u00e9", "barista", "espresso", "expreso", "extracción", "extraccion",
    "molino", "molienda", "tueste", "tostado", "latte", "capuchino", "chemex",
    "v60", "aeropress", "kalita", "percolador", "goteo", "inmersión", "inmersion",
    "taza", "terroir", "altitud", "beneficio", "lavado", "honey", "natural"
]

KEYWORDS = [
    "café", "bebida"
]

def contains_keywords(text: str) -> bool:
    t = (text or "").lower()
    # return any(k in t for k in KEYWORDS)
    return all(k in t for k in KEYWORDS)

# En wikimedia/wikipedia, el campo suele ser 'text'
wiki_filtered = wiki.filter(lambda x: contains_keywords(x.get("text", "")))
print("Total artículos filtrados:", len(wiki_filtered))

Filter:   0%|          | 0/1841155 [00:00<?, ? examples/s]

Total artículos filtrados: 1677


In [8]:
# barajar y ver una muestra
wiki_filtered = wiki_filtered.shuffle(seed=42)
# wiki_filtered = wiki_filtered.shuffle(seed=42).select(range(20000))
# wiki_filtered.select(range(min(3, len(wiki_filtered))))

In [9]:
wiki_filtered

Dataset({
    features: ['id', 'url', 'title', 'text'],
    num_rows: 1677
})

In [10]:
wiki_subset = wiki_filtered.select(range(10))  # o los que quieras

In [11]:
wiki_subset

Dataset({
    features: ['id', 'url', 'title', 'text'],
    num_rows: 10
})

In [12]:
wiki_subset[0]

{'id': '2027055',
 'url': 'https://es.wikipedia.org/wiki/Latte%20macchiato',
 'title': 'Latte macchiato',
 'text': 'No confundir con una bebida similar, caffè macchiato.\nEl latte macchiato es una bebida preparada con leche y café expreso. Latte Macchiato (la(ː)te maˈkja(ː)to) significa en italiano simplemente ‘leche manchada’. El nombre hace referencia al modo de preparación, donde la leche queda «manchada» con el café añadido. Se diferencia de modo significativo del café con leche porque solo se utiliza ½ tiro de expreso (o menos). En sentido estricto, no sería en realidad café sino leche caliente con una mínima proporción de café.\n\nLa mancha es una pequeña mácula en la espuma que queda por encima de la leche para indicar claramente que se trata de un latte macchiato y no un café con leche, donde el café expreso tradicionalmente se añade antes que la leche con lo que no tiene "marca". De forma opuesta, otra bebida similar denominada caffè macchiato es en realidad café expreso manch

In [13]:
wiki_subset[1]

{'id': '6370983',
 'url': 'https://es.wikipedia.org/wiki/Gastronom%C3%ADa%20de%20Chiapa%20de%20Corzo',
 'title': 'Gastronomía de Chiapa de Corzo',
 'text': 'La gastronomía de la ciudad de Chiapa de Corzo cuenta con raíces indígenas (Provenientes de los antiguos pueblos Nahua, Maya, Zoque y Chiapa), españolas, portuguesas, judías y árabe. Dando como resultado una gastronomía muy elaborada y única.\n\nPlatillos principales:\nEntradas:\nEmpanadas chiapacorceñas, lentejas con longaniza y chorizo, picadillo, sopa de fiesta, chipilín con bolitas, memelitas de frijol, agua de chile.\n\nPlatos fuertes:\nPepita con tasajo, cochito horneado, puerco con arroz, chanfaina chiapacorceña, carnes frías, mole almendrado, puerco adobado.\n\nPlatos del diario:\nSalpicón, pellejo con arroz, sopa de pan, cocido, frijol con jocote, frijol con huevo, pellejo con frijol, Tortaditas con recado, caldo de chipilin, estofado, lengua guisada, machaca con huevo o deshebrada, tazajo con chirmol, memelitas\n\nTamales

In [14]:
# del wiki

### ✂️ Limpieza simple y segmentación en fragmentos


In [15]:
import re

def clean_text(t: str) -> str:
    t = re.sub(r"\n{2,}", "\n", t or "")
    t = re.sub(r"={2,}.*?={2,}", " ", t)  # encabezados wiki
    t = re.sub(r"\[\d+\]", "", t)        # refs [1], [2]
    t = re.sub(r"<.*?>", "", t)          # HTML
    return t.strip()

# wiki_filtered = wiki_filtered.map(lambda x: {"text": clean_text(x.get("text",""))})
wiki_subset = wiki_subset.map(lambda x: {"text": clean_text(x.get("text",""))})

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

### 🔗 Tokenización y creación de ejemplos para entrenamiento causal LM


In [16]:
def tokenize_fn(batch):
    return tokenizer(batch["text"], add_special_tokens=False, return_attention_mask=False)

# tokenized = wiki_filtered.map(
tokenized = wiki_subset.map(
    tokenize_fn,
    batched=True,
    # remove_columns=wiki_filtered.column_names,  # quita 'text' y demás
    remove_columns=wiki_subset.column_names,
    batch_size=1000
)

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [17]:
tokenized

Dataset({
    features: ['input_ids'],
    num_rows: 10
})

In [18]:
tokenized[0]

{'input_ids': [2936,
  21551,
  295,
  351,
  16210,
  3124,
  12,
  484,
  2060,
  4149,
  9452,
  22553,
  300,
  14,
  199,
  491,
  276,
  5756,
  9452,
  22553,
  300,
  293,
  351,
  16210,
  25392,
  295,
  10703,
  287,
  8984,
  36963,
  14,
  486,
  5756,
  4595,
  22553,
  300,
  358,
  334,
  8,
  26781,
  9,
  286,
  637,
  35763,
  75,
  820,
  8,
  26781,
  9,
  300,
  9,
  2442,
  278,
  4336,
  5263,
  6659,
  329,
  1342,
  620,
  5180,
  22289,
  462,
  935,
  1623,
  3533,
  336,
  2437,
  258,
  8661,
  12,
  669,
  276,
  10703,
  4836,
  682,
  549,
  5180,
  664,
  295,
  284,
  8984,
  17308,
  14,
  656,
  2999,
  258,
  2437,
  11598,
  309,
  8984,
  295,
  10703,
  1996,
  1084,
  306,
  3760,
  653,
  122,
  7749,
  258,
  36963,
  358,
  79,
  1618,
  600,
  450,
  3438,
  21685,
  12,
  407,
  2161,
  278,
  3300,
  8984,
  2043,
  10703,
  13982,
  295,
  351,
  11271,
  11872,
  258,
  8984,
  14,
  199,
  552,
  26415,
  293,
  351,
  3245,
  1679,
  

In [19]:
from itertools import chain
BLOCK_SIZE = 256

In [20]:
def group_texts(examples):
    # Concatena listas de input_ids dentro del lote, no en toda la colección
    concatenated = list(chain.from_iterable(examples["input_ids"]))
    total_len = (len(concatenated) // BLOCK_SIZE) * BLOCK_SIZE
    if total_len == 0:
        return {"input_ids": []}
    result = {
        "input_ids": [concatenated[i:i+BLOCK_SIZE] for i in range(0, total_len, BLOCK_SIZE)]
    }
    return result

In [21]:
lm_ds = tokenized.map(
    group_texts,
    batched=True,
    batch_size=1000,
    # writer_batch_size controla cuánto se guarda por chunk en disco caché → menos RAM pico
    writer_batch_size=1000
)

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [22]:
lm_ds

Dataset({
    features: ['input_ids'],
    num_rows: 223
})

In [23]:
split = lm_ds.train_test_split(test_size=0.05, seed=42)
train_ds, val_ds = split["train"], split["test"]
train_ds, val_ds

(Dataset({
     features: ['input_ids'],
     num_rows: 211
 }),
 Dataset({
     features: ['input_ids'],
     num_rows: 12
 }))

## 🧩 Data collator (Causal LM, sin máscara MLM)


El **entrenamiento causal de modelos de lenguaje** (*Causal Language Modeling*, CLM) es el método usado por modelos **autoregresivos** como GPT.  
Su objetivo es que el modelo aprenda a **predecir el siguiente token** (palabra o subpalabra) dado el contexto anterior.

---

### 🔹 ¿Cómo funciona?

Durante el entrenamiento, el modelo recibe una secuencia de tokens:

$$
(x_1, x_2, x_3, \dots, x_n)
$$

y debe aprender a maximizar la probabilidad conjunta de toda la secuencia:

$$
P(x_1, x_2, \dots, x_n) = \prod_{t=1}^{n} P(x_t \mid x_{<t})
$$

donde cada token se predice **solo a partir de los anteriores**, no de los futuros.

El modelo ve los tokens en orden y aprende a completar el texto *de izquierda a derecha*.

---

### 🔹 Diferencias con Masked Language Modeling (MLM)

| Característica | **Causal LM (GPT)** | **Masked LM (BERT)** |
|:--|:--|:--|
| Dirección del contexto | Unidireccional (izquierda → derecha) | Bidireccional (ambos lados) |
| Objetivo | Predecir el **siguiente token** | Predecir tokens **enmascarados** |
| Máscara de atención | 🔒 Causal (no ve tokens futuros) | 🔓 Libre (ve todos los tokens) |
| Uso principal | Generación de texto, chat, código | Comprensión, clasificación, embeddings |
| Ejemplo | “El café colombiano es **[?]**” → *aromático* | “El **[MASK]** colombiano es aromático” → *café* |
---
### 🔹 ¿Para qué se utiliza?

Entrenar modelos generativos como GPT, GPT-2, GPT-3, etc.

Afinar modelos en dominios específicos (por ejemplo, lenguaje médico, financiero o de barismo).

Aprender estilos o jerga de un corpus determinado, mejorando la coherencia y la fluidez del texto generado.

Usarse como paso previo para modelos conversacionales o de text completion.



## 🏋️‍♂️ Entrenamiento (Trainer API)


In [24]:
from transformers import DataCollatorForLanguageModeling
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [25]:
from transformers import Trainer, TrainingArguments
import torch, math

In [ ]:
TrainingArguments

In [27]:
OUTPUT_DIR = "gpt2-small-es-cafe-ft"
#BATCH_SIZE = 8
GR_ACCUM = 2
EPOCHS = 10
LR = 5e-5

args = TrainingArguments(
    output_dir=OUTPUT_DIR,              # 📁 Carpeta donde se guardan checkpoints y resultados
#    overwrite_output_dir=True,          # ✅ Permite sobrescribir salidas previas (útil en demos)

    per_device_train_batch_size=8,      # ⚙️ Tamaño de batch por GPU/CPU — pequeño para Colab
    per_device_eval_batch_size=8,       # 🧪 Tamaño de batch para validación

    gradient_accumulation_steps=2,      # 🔄 Acumula gradientes; simula batch más grande sin más RAM
    num_train_epochs=EPOCHS,                 # 🕒 Número de épocas (reducido para clases/demo)
                                         # Puedes subir a 2–3 para ver mejora, pero con más tiempo

    eval_strategy="epoch",              # 🧩 Evalúa al final de cada época (más rápido que cada N pasos)
    save_strategy="epoch",              # 💾 Guarda modelo al final de cada época
    logging_steps=5,                   # 📊 Frecuencia de logs (más bajo = más mensajes)

    learning_rate=5e-5,                 # 🎯 Tasa de aprendizaje estándar para fine-tuning
    lr_scheduler_type="cosine",         # 🔁 Disminuye LR suavemente a lo largo del entrenamiento
    warmup_ratio=0.03,                  # 🔥 Calienta el LR al inicio para evitar oscilaciones

    bf16=torch.cuda.is_available(),     # ⚙️ Usa formato BF16 si la GPU lo soporta (más eficiente)
    fp16=False,                         # ⚠️ Evita FP16 si la GPU no lo soporta bien

    weight_decay=0.01,                  # ⚖️ Regularización ligera para evitar sobreajuste
    report_to="none",                   # 🧾 Desactiva reportes automáticos (wandb, tensorboard)
    push_to_hub=False,                  # 🚫 No sube el modelo al Hugging Face Hub (para demos locales)

    seed=42,                            # 🌱 Reproducibilidad de resultados
)



warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [28]:
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL)
model.resize_token_embeddings(len(tokenizer))

# Alinear especial tokens con el tokenizer
tokenizer.pad_token = tokenizer.eos_token  # para GPT-2 es práctica común

model.config.eos_token_id = tokenizer.eos_token_id
model.config.bos_token_id = getattr(tokenizer, "bos_token_id", None) or tokenizer.eos_token_id
model.config.pad_token_id = tokenizer.pad_token_id

# Alinear también la generación
model.generation_config.eos_token_id = model.config.eos_token_id
model.generation_config.bos_token_id = model.config.bos_token_id
model.generation_config.pad_token_id = model.config.pad_token_id

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: datificate/gpt2-small-spanish
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [29]:
# train_ds = train_ds.select(range(1000))
# val_ds = val_ds.select(range(200))

In [30]:
train_ds

Dataset({
    features: ['input_ids'],
    num_rows: 211
})

In [33]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    # tokenizer=tokenizer,
    processing_class=tokenizer,
)
trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,3.528539,3.113811
2,3.206362,3.079616
3,3.095502,3.048050
4,3.000656,3.032782
5,2.823253,3.029371
6,2.680707,3.027268
7,2.796647,3.024147
8,2.678978,3.023028
9,2.684336,3.022972
10,2.673164,3.022642


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=140, training_loss=2.9104826177869523, metrics={'train_runtime': 471.946, 'train_samples_per_second': 4.471, 'train_steps_per_second': 0.297, 'total_flos': 275663093760000.0, 'train_loss': 2.9104826177869523, 'epoch': 10.0})

## 📏 Evaluación rápida: Perplejidad (val)


In [34]:
eval_res = trainer.evaluate()
val_loss = eval_res["eval_loss"]
ppl = math.exp(val_loss) if val_loss is not None else None
print("Eval results:", eval_res)
print("Perplexity (aprox):", ppl)


Eval results: {'eval_loss': 3.022641897201538, 'eval_runtime': 0.4889, 'eval_samples_per_second': 24.543, 'eval_steps_per_second': 4.09, 'epoch': 10.0}
Perplexity (aprox): 20.545499144088655


## 📊 Métricas para Evaluar Modelos de Lenguaje (LLMs)

La evaluación de modelos de lenguaje es un tema amplio y depende del **tipo de tarea** (modelado del lenguaje, generación abierta, clasificación, resumen, QA, etc.).  
A continuación se resumen las métricas más utilizadas, iniciando con la más fundamental en modelos autoregresivos.

---

# 🧠 1. Perplexity (PPL)

La **Perplejidad** es la métrica clásica para evaluar modelos autoregresivos (GPT, GPT-2, GPT-Neo, etc.).  
Mide qué tan “sorprendido” está un modelo al predecir un conjunto de texto.  
Cuanto **más baja**, mejor.

### Fórmula

Dada una secuencia de tokens $ x_1, x_2, \ldots, x_N $:

$$
\text{PPL} = \exp\left( -\frac{1}{N} \sum_{t=1}^{N} \log P(x_t \mid x_{<t}) \right)
$$

- Si la PPL es baja → el modelo predice bien el texto.  
- Si es alta → el modelo no entiende el dominio.

**Ideal para:**  
✔ Fine-tuning de modelos autoregresivos  
✔ Comparar modelos durante entrenamiento  

---

# 📝 2. Exact Match (EM) y F1 (para Question Answering)

En tareas donde la respuesta debe ser **exacta**, como QA extractivo (SQuAD), se usan:

### 🔹 Exact Match (EM)
Evalúa si la respuesta del modelo **coincide exactamente** con la respuesta correcta.

### 🔹 F1 Score (token-level)
Mide el solapamiento entre tokens de la predicción y la referencia.

**Ideal para:**  
✔ QA extractivo (SQuAD, QA legal, QA en documentos técnicos)

---

# ✍️ 3. BLEU (Machine Translation)

La métrica clásica para traducción automática.

Evalúa coincidencias de **n-gramas** entre la traducción generada y la referencia.

- Mejor para modelos **rule-based** o **estadísticos**.
- Menos útil para LLMs modernos (porque penaliza variaciones creativas).

**Ideal para:**  
✔ Traducción controlada  
✔ Comparar sistemas clásicos vs LLMs

---

# ✍️ 4. ROUGE (Resumen de texto)

ROUGE evalúa cuánto se solapa el resumen generado con el resumen de referencia.

- **ROUGE-1:** Unigramas  
- **ROUGE-2:** Bigramas  
- **ROUGE-L:** Longest Common Subsequence  

**Ideal para:**  
✔ Resumen extractivo/abstractive  
✔ Evaluación de coherencia y cobertura

---

# 📚 5. METEOR

Mejora BLEU incorporando:
- Sinónimos  
- Lematización  
- Alineamiento flexible  

Genera mejor correlación con evaluaciones humanas.

**Ideal para:**  
✔ Traducción  
✔ Parafraseo  
✔ Generación abierta con control lingüístico

---

# 🧩 6. BERTScore

Utiliza **embeddings contextuales** de BERT para comparar la similitud semántica entre el texto generado y el texto objetivo:

- Mucho más robusto que BLEU/ROUGE.  
- Permite variación de superficie (palabras distintas, mismo significado).  

**Ideal para:**  
✔ Tareas generativas donde importa el *significado*, no la exactitud literal  
✔ Evaluar LLMs que producen respuestas más libres  

---

# 🧪 7. MoverScore y BLEURT

Métricas modernas basadas en modelos entrenados para correlacionar con humanos.

### 🔹 BLEURT  
Modelo enseñado explícitamente para puntuar calidad generativa.

### 🔹 MoverScore  
Mide "distancia semántica" entre textos usando Word Mover’s Distance.

**Ideal para:**  
✔ Comparaciones finas entre modelos generativos  
✔ Evaluación de calidad subjetiva

---

# 🗣️ 8. Evaluación Humana (Preferencia / A/B Testing)

Para tareas abiertas (chat, razonamiento, creatividad), las métricas automáticas son insuficientes.

Se usan evaluaciones humanas para medir:

- **Coherencia**
- **Veracidad factual**
- **Estilo**
- **Creatividad**
- **Utilidad**
- **Seguridad**

**Ideal para:**  
✔ Chatbots  
✔ Modelos instructivos  
✔ Generación narrativa  
✔ Asistentes conversacionales  

---

# 🧱 9. Evaluación por Benchmarks Modernos

Para comparar LLMs completos se utilizan suites de pruebas:

### 🔹 MMLU  
Conocimientos generales en 57 áreas.

### 🔹 BIG-Bench  
Evaluación masiva en razonamiento, lógica, matemáticas, commonsense, etc.

### 🔹 HELM (Stanford)  
Marco exhaustivo que evalúa distintos ejes: robustez, equidad, toxicidad, veracidad…

### 🔹 MT-Bench / GPT-Arena  
Comparación de chatbots según preferencia humana.

---

# 🧭 ¿Qué métrica usar en cada caso?

| Tarea | Métrica principal | Alternativas |
|-------|------------------|--------------|
| Modelado del lenguaje | **Perplexity** | – |
| QA extractivo | **EM**, **F1** | – |
| Chat / Dialog | **Evaluación humana**, preferencias | BLEU, ROUGE (indirectas) |
| Traducción | **BLEU**, **METEOR** | BERTScore |
| Resumen | **ROUGE** | BERTScore, BLEURT |
| Parafraseo / NLG general | **BERTScore** | BLEURT, MoverScore |
| Clasificación | Accuracy, F1 | MCC |

---

## ✅ En resumen

> La evaluación de LLMs depende completamente de la tarea.  
> No existe una métrica universal, pero **Perplexity**, **EM/F1**, **BLEU/ROUGE**,  
> y **BERTScore** son las más comunes para medir desempeño de modelos modernos.

Estas métricas permiten comparar modelos, diagnosticar entrenamientos y entender si un fine-tuning realmente **mejora el rendimiento** en la tarea deseada.


## ✍️ Comparación de generaciones (antes vs. después)


In [35]:
from collections import Counter

counter = Counter()

for ids in train_ds["input_ids"]:
    counter.update(ids)

# los 50 tokens más comunes
top_tokens = counter.most_common(100)
[tokenizer.decode([t]) for t,_ in top_tokens]


[' de',
 ',',
 '.',
 ' la',
 ' y',
 ' el',
 ' en',
 '\n',
 ' se',
 ' del',
 ' a',
 ' con',
 ' los',
 ' que',
 ' las',
 ' (',
 ' por',
 ';',
 ' un',
 ' una',
 ' ',
 ' es',
 ' para',
 ' al',
 ' Hidalgo',
 ' El',
 ':',
 ' como',
 ' En',
 ' estado',
 '\xa0',
 ' Pachuca',
 '-',
 'pan',
 ' su',
 ' La',
 ' o',
 ' ra',
 'El',
 'En',
 ')',
 ' más',
 'an',
 ' fue',
 ' son',
 ' %',
 ' Tul',
 'cingo',
 '),',
 'ción',
 'La',
 ' Tula',
 ' México',
 ' no',
 ' C',
 ' entre',
 'utla',
 ' cuenta',
 ');',
 'j',
 ' también',
 ' ciudad',
 ').',
 ' San',
 ' 1',
 ' A',
 ' municipios',
 ' Hu',
 ' "',
 ' Hue',
 ' alcohol',
 ' Valle',
 ' me',
 ' esta',
 '1',
 ' Los',
 ' dos',
 ' municipio',
 '/',
 'to',
 ' este',
 ' durante',
 ' 3',
 '6',
 ' Soto',
 'mi',
 ' Tepe',
 ' sus',
 'cusa',
 ' lo',
 ' parte',
 ' tiene',
 'co',
 ' donde',
 ' Sierra',
 ' Ix',
 ' tres',
 'ol',
 '0',
 ' encuentra']

In [36]:
from transformers import set_seed
import textwrap, torch
set_seed(42)

In [37]:
PROMPTS = [
    "La receta ideal para preparar un V60 es",
    "La diferencia entre el latte macchiato y el café con leche",
    "Los elementos más importantes para prepar un buen café son"
]
gen_kwargs = dict(max_length=80, do_sample=True, temperature=0.9, top_p=0.9, num_return_sequences=1, pad_token_id=tokenizer.eos_token_id)

print("=== BASE MODEL (sin fine-tuning) ===")
base_inputs = tokenizer(PROMPTS[1], return_tensors="pt")
with torch.no_grad():
    base_out = base_model.generate(**base_inputs, **gen_kwargs)
print(textwrap.fill(tokenizer.decode(base_out[0], skip_special_tokens=True), width=100))


=== BASE MODEL (sin fine-tuning) ===
La diferencia entre el latte macchiato y el café con leche es que en el café se consumen café (al
menos el 2% de la masa total del café), y en el café se suelen consumir leche.   Para la preparación
de este café se utiliza la sosa de leche (de la sosa de leche, la sosa de leche o la sosa de leche
sosa


In [38]:
device = trainer.model.device

ft_inputs = tokenizer(PROMPTS[1], return_tensors="pt")
ft_inputs = {k: v.to(device) for k, v in ft_inputs.items()}

gen_kwargs["pad_token_id"] = tokenizer.pad_token_id

# Generación
trainer.model.eval()
with torch.no_grad():
    ft_out = trainer.model.generate(**ft_inputs, **gen_kwargs)

In [39]:
print("\n\n=== FINE-TUNED MODEL (con dominio café) ===")
#ft_inputs = tokenizer(PROMPTS[0], return_tensors="pt")
with torch.no_grad():
    ft_out = trainer.model.generate(**ft_inputs, **gen_kwargs)
print(textwrap.fill(tokenizer.decode(ft_out[0], skip_special_tokens=True), width=100))



=== FINE-TUNED MODEL (con dominio café) ===
La diferencia entre el latte macchiato y el café con leche es que el café con leche contiene una
mayor cantidad de alcohol y es menos soluble en agua que el café con leche. El café con leche
contiene alcohol y solo se da en agua con alcohol de la misma mezcla, lo que no da una proporción
adecuada de alcoholes y alcoholes volátiles. El café con leche contiene alcohol,


## 💾 Guardado local del modelo y tokenizador


In [40]:
SAVE_DIR = "gpt2-small-es-cafe-ft-artifacts"
os.makedirs(SAVE_DIR, exist_ok=True)
trainer.model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Guardado en:", os.path.abspath(SAVE_DIR))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Guardado en: /content/gpt2-small-es-cafe-ft-artifacts


## ☁️ (Opcional) Subir a Hugging Face Hub


In [41]:
# from huggingface_hub import login
# login()  # pega tu token
# trainer.push_to_hub("gpt2-small-es-cafe-ft", private=True)
# tokenizer.push_to_hub("gpt2-small-es-cafe-ft")


## 🧪 Cargar el modelo guardado y verificar generación


In [42]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [43]:
re_tok = AutoTokenizer.from_pretrained("gpt2-small-es-cafe-ft-artifacts")
re_mod = AutoModelForCausalLM.from_pretrained("gpt2-small-es-cafe-ft-artifacts")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [44]:
prompt = "Los elementos más importantes para prepar un buen café son"
inputs = re_tok(prompt, return_tensors="pt")

In [45]:
with torch.no_grad():
    gen = re_tok.decode(re_mod.generate(**inputs, max_length=60, do_sample=True, temperature=0.9, top_p=0.9, pad_token_id=re_tok.eos_token_id)[0], skip_special_tokens=True)
print(gen)

Los elementos más importantes para prepar un buen café son: carne, pollo, cerdo, aves y vegetales. La carne se usa para acompañar carnes como carnes de res, jamón, queso, salchichas y galletas.
Agricultura
En el centro de México se encuentra la producción del café;


In [46]:
with torch.no_grad():
    gen = re_tok.decode(re_mod.generate(**inputs, max_length=60, do_sample=True, temperature=0.5, top_p=0.9, pad_token_id=re_tok.eos_token_id)[0], skip_special_tokens=True)
print(gen)

Los elementos más importantes para prepar un buen café son los sabores, la textura, la textura y la sabor.
En el café, el café es un producto que se prepara con una mezcla de ingredientes naturales como el café, la leche, el cacao, el café y el cacao. El café se


## ✅ Notas
- Aumenta `BLOCK_SIZE`, épocas y tamaño de lote para un *fine-tuning* más notorio (según GPU).
- Usa *gradient checkpointing* o *8-bit optimizers* si te quedas sin memoria.
- Si el `wikipedia` ES no descarga, puedes reemplazar por tu propio `.txt` dominial y reusar el pipeline de tokenización/chunking.
